# Short-term stock market prediction with Time Series Transformer Network	

In [70]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import sampler
import torch.optim as optim

from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd

In [42]:
data_all = pd.read_csv("datasets/Kaggle_Dow_Jones_30_Historical_Data.csv")
data_all.replace('?', np.nan, inplace=True)
data_all

,Date,MMM,American Express,Boeing,Caterpillar,Coca Cola,IBM,Intel,J&J,JPMorgan,McDonalds,Merck & Co.,Microsoft,Procter & Gamble,The Home Depot,Walmart,Walt Disney,DJIA
0,4/1/1999,20.22,25.81,21.70,12.80,16.39,49.38,18.35,26.47,27.39,26.31,31.84,29.28,27.03,26.20,30.97,24.09,9832.51
1,4/5/1999,20.29,26.76,21.94,13.38,16.27,51.31,19.36,26.68,27.18,26.74,31.15,30.00,27.69,26.84,31.68,24.09,10007.33
2,4/6/1999,20.00,26.29,21.98,13.19,16.09,51.05,19.80,26.50,27.35,26.56,31.10,29.72,27.63,26.49,31.78,23.42,9963.49
3,4/7/1999,19.82,26.97,22.30,13.23,15.85,52.03,20.06,26.68,29.12,26.96,31.94,29.48,27.66,27.05,32.39,25.14,10085.31
4,4/8/1999,19.93,27.25,22.14,13.26,16.37,52.17,19.90,27.35,30.33,27.03,32.51,29.88,27.66,28.38,33.96,26.05,10197.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5785,3/28/2022,149.47,189.70,188.16,221.85,61.92,131.47,51.51,177.83,140.87,242.94,81.21,310.70,153.88,314.28,146,138.72,34707.94
5786,3/29/2022,152.08,190.94,193.80,221.04,62.16,131.94,52.25,177.74,141.18,248.17,81.57,315.41,155.46,317.71,147.23,142.38,34861.24
5787,3/30/2022,151.43,189.76,194.91,223.08,62.21,132.13,51.43,179.59,NaN,249.03,82.40,313.86,154.39,308.46,149.87,140.96,34955.89
5788,3/31/2022,148.88,187.00,191.50,222.82,62.00,130.02,49.56,177.23,136.32,247.28,82.05,308.31,152.80,299.33,148.92,137.16,35294.19


In [81]:
window_length=5
pred_length=5
batch_size = 128

#split into train and test datasets
split = 5700
train_data = data_all.iloc[:split, :]
test_data = data_all.iloc[split:, :] 

def preprocess(data, window_length = window_length, pred_length = pred_length):
    #flatten all stocks into one column  
    data = data.iloc[:, 1:].melt()
    data = data.iloc[:, 1]

    data = pd.to_numeric(data).to_numpy() #faster to do slicing operation as numpy array than pandas
    time_length = data.size
    data_windowed = []#np.empty(0)
    for window_index in np.arange( np.random.randint(0,window_length-1), time_length-window_length-pred_length, window_length):  
        window = np.array(data[window_index:window_index+window_length])
        window = window-np.min(window)
        window = window/np.max(window)
        data_windowed.append(window)
        #data_windowed = np.append(data_windowed, window, axis=1)
    
    data_windowed = np.asarray(data_windowed) #convert list to numpy array
    return data_windowed


train_data = preprocess(train_data)
test_data = preprocess(test_data)

train_data = train_data[:, :, None].astype('float32') #expand to include the feature dimension (just 1 for now)
test_data = test_data[:, :, None].astype('float32')
print(train_data.shape)
print(train_data.dtype)

(19378, 5, 1)
float32


In [82]:
train_loader = DataLoader(list(zip(train_data, train_data)), batch_size, shuffle=True) #x is same as y for simple scalar transformers, right?
test_loader = DataLoader(list(zip(test_data, test_data)), batch_size, shuffle=True)

In [83]:
# window_size = 16
# forecast_length = 16
# num_val = 2

# data_all['date'] = pd.to_datetime(data_all['date']) #convert int to datetime
# val_max_date = '2017-08-15'
# train_max_date = str((pd.to_datetime(val_max_date) - pd.Timedelta(days=window_size*num_val+forecast_length)).date())

# train_final = data_all[data_all['date']<=train_max_date]
# val_final = data_all[(data_all['date']>train_max_date)&(data_all['date']<=val_max_date)]

# train_series = train_final.groupby(categorical_static).agg(list).reset_index()
# val_series = val_final.groupby(categorical_static).agg(list).reset_index()

# #try changing date to ints
# train_series
# data_all['date'] = pd.to_datetime(data_all['date']) #convert int to datetime

In [46]:
# def divide_shuffle(df,div_num):
#     space = df.shape[0]//div_num
#     division = np.arange(0,df.shape[0],space)
#     return pd.concat([df.iloc[division[i]:division[i]+space,:].sample(frac=1) for i in range(len(division))])

# def create_time_blocks(time_length,window_size,forecast_length):
#     start_idx = np.random.randint(0,window_size-1)
#     end_idx = time_length-window_size-forecast_length-1
#     time_indices = np.arange(start_idx,end_idx+1,window_size)[:-1]
#     time_indices = np.append(time_indices,end_idx)
#     return time_indices

# def data_loader(x_numeric_tensor, x_category_tensor, x_static_tensor, y_tensor, batch_size, time_shuffle):

#     num_series = x_numeric_tensor.shape[0]
#     time_length = x_numeric_tensor.shape[1]
#     index_pd = pd.DataFrame({'serie_idx':range(num_series)})
#     index_pd['time_idx'] = [create_time_blocks(time_length,window_size,forecast_length) for n in range(index_pd.shape[0])]
#     if time_shuffle:
#         index_pd = index_pd.explode('time_idx')
#         index_pd = index_pd.sample(frac=1)
#     else:
#         index_pd = index_pd.explode('time_idx').sort_values('time_idx')
#         index_pd = divide_shuffle(index_pd,5)
#     indices = np.array(index_pd).astype(int)

#     for batch_idx in np.arange(0,indices.shape[0],batch_size):

#         cur_indices = indices[batch_idx:batch_idx+batch_size,:]

#         x_numeric = torch.stack([x_numeric_tensor[n[0],n[1]:n[1]+window_size,:] for n in cur_indices])
#         x_category = torch.stack([x_category_tensor[n[0],n[1]:n[1]+window_size,:] for n in cur_indices])
#         x_static = torch.stack([x_static_tensor[n[0],:] for n in cur_indices])
#         y = torch.stack([y_tensor[n[0],n[1]+window_size:n[1]+window_size+forecast_length] for n in cur_indices])

#         yield x_numeric.to(device), x_category.to(device), x_static.to(device), y.to(device)

# def val_loader(x_numeric_tensor, x_category_tensor, x_static_tensor, y_tensor, batch_size, num_val):

#     num_time_series = x_numeric_tensor.shape[0]

#     for i in range(num_val):

#       for batch_idx in np.arange(0,num_time_series,batch_size):

#           x_numeric = x_numeric_tensor[batch_idx:batch_idx+batch_size,window_size*i:window_size*(i+1),:]
#           x_category = x_category_tensor[batch_idx:batch_idx+batch_size,window_size*i:window_size*(i+1),:]
#           x_static = x_static_tensor[batch_idx:batch_idx+batch_size]
#           y_val = y_tensor[batch_idx:batch_idx+batch_size,window_size*(i+1):window_size*(i+1)+forecast_length]

#           yield x_numeric.to(device), x_category.to(device), x_static.to(device), y_val.to(device)

In [47]:
class transformer_block(nn.Module):

    def __init__(self,embed_size,num_heads):
        super(transformer_block, self).__init__()

        self.attention = nn.MultiheadAttention(embed_size, num_heads, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(embed_size, 4 * embed_size),
                                 nn.LeakyReLU(),
                                 nn.Linear(4 * embed_size, embed_size))
        self.dropout = nn.Dropout(drop_prob)
        self.ln1 = nn.LayerNorm(embed_size, eps=1e-6)
        self.ln2 = nn.LayerNorm(embed_size, eps=1e-6)

    def forward(self, x):

        attn_out, _ = self.attention(x, x, x, need_weights=False)
        x = x + self.dropout(attn_out)
        x = self.ln1(x)

        fc_out = self.fc(x)
        x = x + self.dropout(fc_out)
        x = self.ln2(x)

        return x


class transformer_forecaster(nn.Module):

    def __init__(self,embed_size,num_heads,num_blocks):
        super(transformer_forecaster, self).__init__()

        num_len = len(numeric_covariates)
        self.embedding_cov = nn.ModuleList([nn.Embedding(n,embed_size-num_len) for n in categorical_covariates_num_embeddings])
        self.embedding_static = nn.ModuleList([nn.Embedding(n,embed_size-num_len) for n in categorical_static_num_embeddings])

        self.blocks = nn.ModuleList([transformer_block(embed_size,num_heads) for n in range(num_blocks)])

        self.forecast_head = nn.Sequential(nn.Linear(embed_size, embed_size*2),
                                           nn.LeakyReLU(),
                                           nn.Dropout(drop_prob),
                                           nn.Linear(embed_size*2, embed_size*4),
                                           nn.LeakyReLU(),
                                           nn.Linear(embed_size*4, forecast_length),
                                           nn.ReLU())

    def forward(self, x_numeric, x_category, x_static):

        tmp_list = []
        for i,embed_layer in enumerate(self.embedding_static):
            tmp_list.append(embed_layer(x_static[:,i]))
        categroical_static_embeddings = torch.stack(tmp_list).mean(dim=0).unsqueeze(1)

        tmp_list = []
        for i,embed_layer in enumerate(self.embedding_cov):
            tmp_list.append(embed_layer(x_category[:,:,i]))
        categroical_covariates_embeddings = torch.stack(tmp_list).mean(dim=0)
        T = categroical_covariates_embeddings.shape[1]

        embed_out = (categroical_covariates_embeddings + categroical_static_embeddings.repeat(1,T,1))/2
        x = torch.concat((x_numeric,embed_out),dim=-1)

        for block in self.blocks:
            x = block(x)

        x = x.mean(dim=1)
        x = self.forecast_head(x)

        return x
    



class RMSLELoss(nn.Module):

    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, pred, actual):
        return torch.sqrt(self.mse(torch.log(pred + 1), torch.log(actual + 1)))

In [84]:
#Pytorch vanilla transformer

transformer_model = nn.Transformer(d_model = 1, nhead=1, num_encoder_layers=6, num_decoder_layers=6, batch_first=True) #only 1 head bc of 1 input dimension
src = torch.rand((batch_size, window_length, 1))#10, 32, 512)) #replace w ...
tgt = torch.rand((batch_size, pred_length, 1))#20, 32, 512))
print(src.dtype)
out = transformer_model(src, tgt)

torch.float32


c:\Users\ebook\Documents\1) School\0) UCSD\ECE 176\Final Project\.venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


In [ ]:
#train the damn thing

num_epoch = 1000
min_val_loss = 999
data_folder = ''
num_blocks = 1
embed_size = 5 #00
num_heads = 8
learning_rate = 3e-4
time_shuffle = False
drop_prob = 0.1

model = transformer_model
criterion = RMSLELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

for epoch in range(num_epoch):
    
    model.train()
    train_loss = 0
    counter = 0
    mask = nn.Transformer.generate_square_subsequent_mask(sz=window_length) #generate casual mask
    
    for x_batch, y_batch in train_loader: #need y tensor too?
        #print(x_batch.size())
        #print(x_batch[0])
        optimizer.zero_grad()
        
        pred = model(x_batch, y_batch, src_mask=mask, tgt_mask=mask, src_is_causal=True, tgt_is_causal=True)  #is the the correct place to enter the forward() parameters?
        loss = criterion(pred, y_batch)
        train_loss += loss.item()
        counter += 1
        loss.backward()
        optimizer.step()

    train_loss = train_loss/counter
    print(f'Epoch {epoch} training loss: {train_loss}')

    model.eval()
    val_loss = 0
    counter = 0
    for x_batch, y_batch in test_loader:
        with torch.no_grad():
            pred = model(x_batch, y_batch, src_is_causal=True, tgt_is_causal=True)
            loss = criterion(pred,y_batch).item()
        val_loss += loss
        counter += 1
    val_loss = val_loss/counter
    print(f'Epoch {epoch} validation loss: {val_loss}')

    if val_loss<min_val_loss:
      print('saved...')
      torch.save(model,data_folder+'best.model')
      min_val_loss = val_loss

    scheduler.step()

torch.Size([128, 5, 1])
tensor([[0.4908],
        [0.1779],
        [0.0000],
        [0.6810],
        [1.0000]])
torch.Size([128, 5, 1])
tensor([[0.1818],
        [1.0000],
        [0.0682],
        [0.0000],
        [0.2803]])
torch.Size([128, 5, 1])
tensor([[0.0769],
        [0.3846],
        [0.3462],
        [0.0000],
        [1.0000]])
torch.Size([128, 5, 1])
tensor([[0.1646],
        [0.6456],
        [0.0000],
        [0.2911],
        [1.0000]])
torch.Size([128, 5, 1])
tensor([[0.3827],
        [0.8642],
        [1.0000],
        [0.0000],
        [0.2099]])
torch.Size([128, 5, 1])
tensor([[1.0000],
        [0.5417],
        [0.6042],
        [0.3750],
        [0.0000]])
torch.Size([128, 5, 1])
tensor([[0.0000],
        [0.4138],
        [0.7759],
        [1.0000],
        [0.8103]])
torch.Size([128, 5, 1])
tensor([[0.9583],
        [1.0000],
        [0.9583],
        [0.8000],
        [0.0000]])
torch.Size([128, 5, 1])
tensor([[0.8824],
        [1.0000],
        [0.4412],
  

KeyboardInterrupt: 